# DEE — Relaciones de Kramers-Kronig

Verificamos que la susceptibilidad del sustrato satisface las relaciones de Kramers-Kronig (consecuencia de causalidad lineal), comparamos con Debye, y exploramos si la susceptibilidad local depende de la presencia de defectos.

**Tres tests:**

1. **Auto-consistencia KK en cristal perfecto:** χ'(ω) calculado directamente debe coincidir con el obtenido integrando χ''(ω) via KK.

2. **Comparación con Debye puro:** desviaciones indican estructura fonónica del sustrato.

3. **Susceptibilidad local con defecto:** χ(ω) visto desde dentro de una inclusión blanda vs desde el cristal lejano. Deben diferir en la región del estado ligado.

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_KK'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

In [ ]:
# Funciones base
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian(points, k_neighbors=30, alpha=2.0, L=1.0, force_modifiers=None):
    D_mat = periodic_distance_matrix(points, L=L)
    N = len(points)
    if force_modifiers is None: force_modifiers = {}
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                w = 1.0 / d**alpha
                mod_i = force_modifiers.get(i, 1.0)
                mod_j = force_modifiers.get(j, 1.0)
                w = w * np.sqrt(mod_i * mod_j)
                rows.append(i); cols.append(j); vals.append(w)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W

# Construcción del grafo y del cristal perfecto
N_TARGET = 1331
K_NEIGHBORS = 30
JITTER = 0.1

points, N = grid_perturbed_T3(N_TARGET, JITTER, seed=42)
L_perfect = build_laplacian(points, K_NEIGHBORS, 2.0)

# Calculamos muchos autovalores para buena resolución en ω
N_EIGS = 400
print(f'Diagonalizando cristal perfecto (k={N_EIGS})...')
t0 = time.time()
eigs_perfect, vecs_perfect = eigsh(L_perfect, k=N_EIGS, which='SM', tol=1e-8)
idx = np.argsort(eigs_perfect)
eigs_perfect = eigs_perfect[idx]
vecs_perfect = vecs_perfect[:, idx]
print(f'  {time.time()-t0:.1f}s')

eigs_nonzero = eigs_perfect[eigs_perfect > 1e-3]
vecs_nonzero = vecs_perfect[:, eigs_perfect > 1e-3]
omegas_perfect = np.sqrt(eigs_nonzero)
print(f'Modos útiles: {len(omegas_perfect)}')
print(f'Rango ω: [{omegas_perfect.min():.3f}, {omegas_perfect.max():.3f}]')

## Test 1 — Auto-consistencia de Kramers-Kronig

Calculamos χ'(ω) y χ''(ω) directamente del espectro, después usamos KK para reconstruir χ'(ω) a partir de χ''(ω) integrando numéricamente. Si el sustrato es causal lineal (lo es por construcción), ambos deben coincidir.

**Susceptibilidad global (promedio sobre todos los nodos):**

$$\chi(\omega) = \frac{1}{N} \sum_n \frac{1}{\omega_n^2 - \omega^2 - i\epsilon}$$

Usamos un ε pequeño (ancho de Lorentziano) para regularizar las delta funciones. Esto ensancha los picos pero mantiene las relaciones KK exactas.

In [ ]:
# Grilla de frecuencias
omega_min = omegas_perfect.min() * 0.5
omega_max = omegas_perfect.max() * 1.5
n_grid = 500
omegas_grid = np.linspace(omega_min, omega_max, n_grid)

# Regularizador (ancho Lorentziano)
# Debe ser mayor que el spacing de autovalores para que las curvas sean suaves
# pero menor que la escala física para preservar estructura
epsilon = 0.5

def susceptibility_direct(omegas_grid, omega_n, weights=None, epsilon=0.5):
    """
    χ(ω) = Σ_n w_n / (ω_n² - ω² - iε · ω)
    Con regularizador causal iε·ω (garantiza KK exacto).
    """
    if weights is None:
        weights = np.ones_like(omega_n) / len(omega_n)
    chi = np.zeros(len(omegas_grid), dtype=complex)
    for w_n, weight in zip(omega_n, weights):
        # Forma causal: pole en ω = ω_n - iε/2 y ω = -ω_n - iε/2
        chi += weight / (w_n**2 - omegas_grid**2 - 1j * epsilon * omegas_grid)
    return chi

# Calcular χ directamente
chi_direct = susceptibility_direct(omegas_grid, omegas_perfect, epsilon=epsilon)
chi_real = chi_direct.real
chi_imag = chi_direct.imag

print(f'Susceptibilidad calculada en {n_grid} puntos de ω')
print(f'Rango χ_real: [{chi_real.min():.3f}, {chi_real.max():.3f}]')
print(f'Rango χ_imag: [{chi_imag.min():.3f}, {chi_imag.max():.3f}]')

In [ ]:
# Reconstruir χ'(ω) a partir de χ''(ω) usando Kramers-Kronig
# χ'(ω) = (2/π) P ∫ [ω' · χ''(ω') / (ω'² - ω²)] dω'

def kk_reconstruct_real(omegas_grid, chi_imag):
    """
    Reconstruye χ'(ω) a partir de χ''(ω) via Kramers-Kronig.
    Usa valor principal de Cauchy con pequeño desplazamiento.
    """
    chi_real_reconstructed = np.zeros_like(omegas_grid)
    for i, omega in enumerate(omegas_grid):
        # Integrando: 2/π · ω' · χ''(ω') / (ω'² - ω²)
        integrand = 2/np.pi * omegas_grid * chi_imag / (omegas_grid**2 - omega**2 + 1e-12)
        # Excluir un entorno del polo ω' = ω (valor principal)
        mask = np.abs(omegas_grid - omega) > 2 * (omegas_grid[1] - omegas_grid[0])
        chi_real_reconstructed[i] = np.trapz(integrand[mask], omegas_grid[mask])
    return chi_real_reconstructed

print('Reconstruyendo χ\' via Kramers-Kronig...')
t0 = time.time()
chi_real_kk = kk_reconstruct_real(omegas_grid, chi_imag)
print(f'  {time.time()-t0:.1f}s')

# Comparación cuantitativa
# Nos enfocamos en un rango donde el efecto KK es robusto (evitamos bordes)
mask_valid = (omegas_grid > omega_min + 2) & (omegas_grid < omega_max - 2)
# Ignoramos la zona cerca de los polos del espectro

error_abs = chi_real_kk[mask_valid] - chi_real[mask_valid]
error_rel = error_abs / (np.abs(chi_real[mask_valid]) + 1e-6)

print(f'\nComparación χ\' directo vs χ\' vía KK:')
print(f'  Error absoluto medio: {np.mean(np.abs(error_abs)):.4f}')
print(f'  Error absoluto máx:   {np.max(np.abs(error_abs)):.4f}')
print(f'  Error relativo medio: {np.mean(np.abs(error_rel))*100:.2f}%')

In [ ]:
# Plot Test 1
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Panel 1: χ'(ω) y χ''(ω) directos
ax = axes[0, 0]
ax.plot(omegas_grid, chi_real, 'b-', lw=2, label="χ'(ω) directo")
ax.plot(omegas_grid, chi_imag, 'r-', lw=2, label="χ''(ω) directo")
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('ω')
ax.set_ylabel('χ(ω)')
ax.set_title('Susceptibilidad del cristal perfecto')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Panel 2: comparación χ' directo vs reconstruido por KK
ax = axes[0, 1]
ax.plot(omegas_grid, chi_real, 'b-', lw=2.5, label="χ'(ω) directo")
ax.plot(omegas_grid, chi_real_kk, '--', color='orange', lw=1.5,
        label="χ'(ω) reconstruido por KK")
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('ω')
ax.set_ylabel("χ'(ω)")
ax.set_title("Test 1: ¿χ' directo coincide con χ' reconstruido por KK?")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Panel 3: error absoluto
ax = axes[1, 0]
ax.plot(omegas_grid, chi_real - chi_real_kk, 'g-', lw=1.5)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('ω')
ax.set_ylabel("χ'_directo − χ'_KK")
ax.set_title('Residuos de la relación Kramers-Kronig')
ax.grid(alpha=0.3)

# Panel 4: error relativo en zona válida
ax = axes[1, 1]
rel_err_full = np.zeros_like(omegas_grid)
rel_err_full[mask_valid] = error_rel
ax.plot(omegas_grid[mask_valid], error_rel * 100, 'm-', lw=1.5)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('ω')
ax.set_ylabel('Error relativo [%]')
ax.set_title('Error relativo en la zona válida')
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('29_KK_autoconsistencia')
plt.show()

## Test 2 — Comparación con Debye puro

En el modelo de Debye, g(ω) ∝ ω² hasta la frecuencia ω_D. La susceptibilidad toma forma cerrada:

$$\chi''_{\text{Debye}}(\omega) \propto \omega \quad (0 < \omega < \omega_D)$$

Comparamos con χ''(ω) observado y vemos cuánto se desvía.

In [ ]:
# Frecuencia de Debye y amplitud de ajuste
omega_D = omegas_perfect[int(0.9 * len(omegas_perfect))]

# Debye puro: χ''(ω) = A · ω para ω < ω_D, 0 en otro caso
# Normalizamos A para que el pico de Debye coincida con el pico máximo observado
mask_below_D = (omegas_grid > 0) & (omegas_grid < omega_D)
max_chi_imag_obs = chi_imag[mask_below_D].max() if mask_below_D.sum() > 0 else 1.0
A_debye = max_chi_imag_obs / omega_D
chi_imag_debye = np.where((omegas_grid > 0) & (omegas_grid < omega_D),
                            A_debye * omegas_grid, 0.0)

# Plot comparación
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(omegas_grid, chi_imag, 'r-', lw=2, label="χ''(ω) sustrato DEE")
ax.plot(omegas_grid, chi_imag_debye, '--', color='gray', lw=2,
        label=f'Debye puro (ω_D = {omega_D:.1f})')
ax.axvline(omega_D, color='black', linestyle=':', alpha=0.5)
ax.set_xlabel('ω')
ax.set_ylabel("χ''(ω)")
ax.set_title('χ\'\'(ω): sustrato real vs Debye puro')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Desviación relativa de Debye
ax = axes[1]
# Ratio observado / Debye
mask_both = (chi_imag_debye > 0.01 * max_chi_imag_obs)
ratio = np.zeros_like(omegas_grid)
ratio[mask_both] = chi_imag[mask_both] / chi_imag_debye[mask_both]
ax.plot(omegas_grid[mask_both], ratio[mask_both], 'g-', lw=2)
ax.axhline(1.0, color='gray', linestyle='--', label='Debye = 1')
ax.set_xlabel('ω')
ax.set_ylabel('χ\'\'_sustrato / χ\'\'_Debye')
ax.set_title('Desviación de Debye — picos = estructura fonónica')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('30_comparacion_Debye')
plt.show()

## Test 3 — Susceptibilidad local con defecto

La susceptibilidad local vista desde un nodo i específico es:

$$\chi_i(\omega) = \sum_n \frac{|v_n(i)|^2}{\omega_n^2 - \omega^2 - i\epsilon \omega}$$

donde |v_n(i)|² es la amplitud del modo n en el nodo i. Si un modo está localizado cerca del nodo, contribuye mucho a χ_i; si está lejos, contribuye poco.

Construimos el cristal con una inclusión 3D blanda (la que localizó más fuerte en el test anterior) y comparamos χ visto desde:
- Un nodo dentro de la inclusión
- Un nodo lejos en el cristal ordinario

In [ ]:
# Inclusión blanda (misma que el test de defectos blandos)
center = np.array([0.5, 0.5, 0.5])
diffs = points - center; diffs -= np.round(diffs)
dist_center = np.linalg.norm(diffs, axis=1)
inclusion_nodes = np.where(dist_center < 0.15)[0]

INCL_SOFT = 0.1
mods_incl = {int(i): INCL_SOFT for i in inclusion_nodes}

L_incl = build_laplacian(points, K_NEIGHBORS, 2.0, force_modifiers=mods_incl)

t0 = time.time()
eigs_incl, vecs_incl = eigsh(L_incl, k=300, which='SM', tol=1e-8)
idx = np.argsort(eigs_incl)
eigs_incl = eigs_incl[idx]
vecs_incl = vecs_incl[:, idx]
print(f'Diagonalización con inclusión: {time.time()-t0:.1f}s')

eigs_incl_nz = eigs_incl[eigs_incl > 1e-3]
vecs_incl_nz = vecs_incl[:, eigs_incl > 1e-3]
omegas_incl = np.sqrt(eigs_incl_nz)

In [ ]:
# Elegir dos nodos: uno dentro de la inclusión (cerca del centro), otro lejos
# Nodo interior: el más cercano al centro
node_inside = int(inclusion_nodes[np.argmin(dist_center[inclusion_nodes])])

# Nodo exterior: el más lejano del centro pero no en un borde del cubo
mask_exterior = dist_center > 0.4
exterior_candidates = np.where(mask_exterior)[0]
node_outside = int(exterior_candidates[0])  # cualquiera suficientemente lejos

print(f'Nodo INTERIOR: índice {node_inside}, pos {points[node_inside]}')
print(f'  Distancia al centro: {dist_center[node_inside]:.3f}')
print(f'Nodo EXTERIOR: índice {node_outside}, pos {points[node_outside]}')
print(f'  Distancia al centro: {dist_center[node_outside]:.3f}')

def local_susceptibility(omegas_grid, omega_n, eigenvectors, node_idx, epsilon=0.5):
    """Susceptibilidad local en un nodo específico."""
    chi = np.zeros(len(omegas_grid), dtype=complex)
    for m, w_n in enumerate(omega_n):
        amp_sq = eigenvectors[node_idx, m]**2
        chi += amp_sq / (w_n**2 - omegas_grid**2 - 1j * epsilon * omegas_grid)
    return chi

# Grilla más fina en bajas frecuencias donde está el estado ligado
omegas_fine = np.linspace(8, 30, 400)

chi_inside = local_susceptibility(omegas_fine, omegas_incl, vecs_incl_nz, node_inside, epsilon=0.3)
chi_outside = local_susceptibility(omegas_fine, omegas_incl, vecs_incl_nz, node_outside, epsilon=0.3)

# Por comparación, el cristal perfecto visto desde el mismo nodo interior
chi_perfect_inside = local_susceptibility(omegas_fine, omegas_perfect, vecs_nonzero, node_inside, epsilon=0.3)

print(f'\nSusceptibilidades calculadas en {len(omegas_fine)} puntos')

In [ ]:
# Plot Test 3
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Panel 1: χ'' interior vs exterior vs cristal perfecto
ax = axes[0, 0]
ax.plot(omegas_fine, chi_inside.imag, 'r-', lw=2.5, label="Interior (dentro de inclusión)")
ax.plot(omegas_fine, chi_outside.imag, 'b-', lw=2, label="Exterior (lejos del defecto)")
ax.plot(omegas_fine, chi_perfect_inside.imag, '--', color='gray', lw=1.5,
        label="Cristal perfecto (mismo nodo)")
ax.axvline(omegas_incl[0], color='red', linestyle=':', alpha=0.5,
           label=f'ω mínimo con inclusión ({omegas_incl[0]:.2f})')
ax.axvline(omegas_perfect[0], color='blue', linestyle=':', alpha=0.5,
           label=f'ω mínimo cristal perfecto ({omegas_perfect[0]:.2f})')
ax.set_xlabel('ω')
ax.set_ylabel("χ''(ω)")
ax.set_title("χ''(ω) local — ¿el defecto cambia la respuesta?")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: mismo pero en log
ax = axes[0, 1]
ax.semilogy(omegas_fine, np.abs(chi_inside.imag) + 1e-6, 'r-', lw=2.5, label='Interior')
ax.semilogy(omegas_fine, np.abs(chi_outside.imag) + 1e-6, 'b-', lw=2, label='Exterior')
ax.semilogy(omegas_fine, np.abs(chi_perfect_inside.imag) + 1e-6, '--', color='gray',
            lw=1.5, label='Cristal perfecto')
ax.set_xlabel('ω')
ax.set_ylabel("|χ''(ω)| (log)")
ax.set_title("Escala log: detalle de estados localizados")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

# Panel 3: χ' interior vs exterior
ax = axes[1, 0]
ax.plot(omegas_fine, chi_inside.real, 'r-', lw=2.5, label="Interior")
ax.plot(omegas_fine, chi_outside.real, 'b-', lw=2, label="Exterior")
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('ω')
ax.set_ylabel("χ'(ω)")
ax.set_title("χ'(ω) local — parte reactiva")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Panel 4: diferencia interior - exterior
ax = axes[1, 1]
ax.plot(omegas_fine, chi_inside.imag - chi_outside.imag, 'g-', lw=2)
ax.axhline(0, color='black', lw=0.5)
ax.axvline(omegas_incl[0], color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('ω')
ax.set_ylabel("χ''_interior − χ''_exterior")
ax.set_title('Diferencia: modos del defecto visibles desde dentro')
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('31_susceptibilidad_local_defecto')
plt.show()

In [ ]:
# Análisis cuantitativo del Test 3
print('='*70)
print('ANÁLISIS — Susceptibilidad local con defecto')
print('='*70)

# Primera frecuencia propia del sistema con defecto
omega_estado_ligado = omegas_incl[0]
idx_bound_in_grid = int(np.argmin(np.abs(omegas_fine - omega_estado_ligado)))

chi_inside_at_bound = chi_inside.imag[idx_bound_in_grid]
chi_outside_at_bound = chi_outside.imag[idx_bound_in_grid]

print(f'\nEstado ligado a ω = {omega_estado_ligado:.3f}:')
print(f'  χ\'\' visto desde interior: {chi_inside_at_bound:.4f}')
print(f'  χ\'\' visto desde exterior: {chi_outside_at_bound:.4f}')
if abs(chi_outside_at_bound) > 1e-6:
    ratio = chi_inside_at_bound / chi_outside_at_bound
    print(f'  Ratio interior/exterior:  {ratio:.1f}x')
    
    if ratio > 10:
        print(f'  → El estado ligado es MUCHÍSIMO más visible desde dentro')
        print(f'     Confirma localización cuantitativa vista a través de KK')
    elif ratio > 3:
        print(f'  → El estado ligado es más visible desde dentro')
    else:
        print(f'  → Diferencia menor — localización débil')

# Integral de χ'' (regla de suma)
integral_inside = np.trapz(chi_inside.imag, omegas_fine)
integral_outside = np.trapz(chi_outside.imag, omegas_fine)

print(f'\nIntegral de χ\'\' (regla de suma f):')
print(f'  Interior: {integral_inside:.3f}')
print(f'  Exterior: {integral_outside:.3f}')
print(f'  (deben ser iguales si ambos nodos tienen misma masa efectiva)')

## Síntesis

In [ ]:
print('='*70)
print('SÍNTESIS — Kramers-Kronig en el sustrato DEE')
print('='*70)
print()

# Test 1
err_KK_medio = np.mean(np.abs(error_rel))*100
print(f'TEST 1 — Auto-consistencia KK:')
print(f'  Error relativo medio: {err_KK_medio:.1f}%')
if err_KK_medio < 10:
    print(f'  → ✓ KK se satisface → sustrato es causal lineal (como debía ser)')
elif err_KK_medio < 25:
    print(f'  → ~ KK se satisface parcialmente (probables efectos de borde de espectro truncado)')
else:
    print(f'  → ✗ KK se viola — sistema tiene propiedades no causales (inesperado)')

print()
print(f'TEST 2 — Desviación de Debye:')
# Calcular desviación cuadrática promedio de Debye
deviation = np.mean((ratio[mask_both] - 1)**2)
print(f'  Desviación cuadrática promedio del ratio Debye: {deviation:.3f}')
print(f'  (0 = Debye puro, >0.5 = estructura fonónica fuerte)')

print()
print(f'TEST 3 — Susceptibilidad local con defecto:')
if abs(chi_outside_at_bound) > 1e-6:
    ratio_local = chi_inside_at_bound / chi_outside_at_bound
    print(f'  Amplificación interior/exterior en estado ligado: {ratio_local:.1f}x')
    if ratio_local > 10:
        print(f'  → ✓✓ Fuerte dependencia espacial de χ confirma localización vía KK')
    elif ratio_local > 3:
        print(f'  → ✓ Dependencia espacial moderada')

print()
print('Implicaciones:')
print('- Sustrato DEE satisface causalidad lineal (Kramers-Kronig).')
print('- La DoS del sustrato tiene estructura fonónica (no es Debye puro).')
print('- La susceptibilidad local responde fuertemente a la presencia de defectos:')
print('  un observador dentro de una región blanda ve estados disponibles que')
print('  uno exterior no percibe.')

In [ ]:
import shutil
shutil.make_archive('plots_KK', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_KK.zip')

try:
    from google.colab import files
    files.download('plots_KK.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass